# 01 — Data Exploration: AML Sample Dataset

EDA evidence over  (synthetic, canonical schema, fixed seed 42).  
Used by judges and reviewers to verify dataset quality and pattern cohort integrity.

**Do not edit this notebook manually** — re-run from top to regenerate all outputs.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

SAMPLE_TX   = Path("/Users/kalyan/AML_67/data/sample/aml_sample.csv")
SAMPLE_CUST = Path("/Users/kalyan/AML_67/data/sample/aml_sample_customers.csv")

tx   = pd.read_csv(SAMPLE_TX, parse_dates=["timestamp"])
cust = pd.read_csv(SAMPLE_CUST)

print(f"Transactions : {len(tx):,} rows × {len(tx.columns)} cols")
print(f"Customers    : {len(cust):,} rows × {len(cust.columns)} cols")


Transactions : 2,002 rows × 13 cols
Customers    : 270 rows × 10 cols


## 1. Schema Compliance Check

In [2]:
TX_REQUIRED = [
    "txn_id", "timestamp", "sender_id", "receiver_id", "amount", "currency",
    "txn_type", "channel", "sender_country", "receiver_country", "is_cross_border",
    "label_is_laundering", "pattern_label",
]
CUST_REQUIRED = [
    "customer_id", "name", "account_open_date", "customer_type", "country",
    "occupation", "risk_rating", "kyc_status", "is_pep", "expected_monthly_volume",
]

missing_tx   = set(TX_REQUIRED)   - set(tx.columns)
missing_cust = set(CUST_REQUIRED) - set(cust.columns)
extra_tx     = set(tx.columns)   - set(TX_REQUIRED)
extra_cust   = set(cust.columns) - set(CUST_REQUIRED)

print("TX  missing:", missing_tx   or "NONE")
print("TX  extra:  ", extra_tx     or "NONE")
print("CUST missing:", missing_cust or "NONE")
print("CUST extra:  ", extra_cust   or "NONE")
print()
print("TX dtypes:")
print(tx.dtypes)


TX  missing: NONE
TX  extra:   NONE
CUST missing: NONE
CUST extra:   NONE

TX dtypes:
txn_id                         object
timestamp              datetime64[ns]
sender_id                      object
receiver_id                    object
amount                        float64
currency                       object
txn_type                       object
channel                        object
sender_country                 object
receiver_country               object
is_cross_border                  bool
label_is_laundering            object
pattern_label                  object
dtype: object


## 2. Row Counts and Class Balance

In [3]:
n_total = len(tx)
n_labelled = tx["label_is_laundering"].notna().sum()
n_pos = tx["label_is_laundering"].fillna(False).astype(bool).sum()
n_normal = tx["pattern_label"].isna().sum()

print(f"Total transactions    : {n_total:,}")
print(f"Normal (unlabelled)   : {n_normal:,} ({100*n_normal/n_total:.1f}%)")
print(f"Labelled laundering   : {n_pos:,} ({100*n_pos/n_total:.1f}%)")
print()

# Pattern cohort sizes
print("Pattern cohort breakdown:")
pattern_counts = tx["pattern_label"].value_counts(dropna=False)
print(pattern_counts.to_string())
print()

# Customer counts
n_cust = len(cust)
print(f"Total customers       : {n_cust:,}")
print(cust["customer_type"].value_counts().to_string())
print()
print("Risk rating:")
print(cust["risk_rating"].value_counts().to_string())


Total transactions    : 2,002
Normal (unlabelled)   : 1,800 (89.9%)
Labelled laundering   : 202 (10.1%)

Pattern cohort breakdown:
pattern_label
NaN              1800
structuring        91
smurfing           51
rapid_cashout      40
layering           20

Total customers       : 270
customer_type
individual    204
business       66

Risk rating:
risk_rating
low       217
medium     39
high       14


### 2a. Class balance chart

In [4]:
fig = px.pie(
    values=[n_normal, n_pos],
    names=["Normal", "Laundering"],
    title="Transaction Class Balance (synthetic aml_sample.csv)",
    color_discrete_sequence=["#4CAF50", "#F44336"],
)
fig.update_traces(textinfo="percent+value")
fig.show()


### 2b. Pattern cohort bar chart

In [5]:
pattern_df = tx["pattern_label"].value_counts(dropna=True).reset_index()
pattern_df.columns = ["pattern", "count"]
fig = px.bar(
    pattern_df,
    x="pattern", y="count",
    title="AML Pattern Cohort Sizes",
    color="pattern",
    color_discrete_map={
        "structuring": "#FF5722",
        "smurfing": "#9C27B0",
        "layering": "#2196F3",
        "rapid_cashout": "#FF9800",
    },
    text="count",
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, xaxis_title="Pattern", yaxis_title="Transactions")
fig.show()


## 3. Amount Distribution

In [6]:
print("Amount statistics:")
print(tx["amount"].describe().round(2))
print()
print("Amount stats by pattern:")
print(tx.groupby(tx["pattern_label"].fillna("normal"))["amount"].describe().round(2))


Amount statistics:
count      2002.00
mean      10133.86
std       27464.68
min          51.38
25%        4025.58
50%        7893.30
75%       11233.51
max      460311.40
Name: amount, dtype: float64

Amount stats by pattern:
                count       mean        std       min       25%        50%        75%        max
pattern_label                                                                                   
layering         20.0  218590.83  163812.04  31498.02  55027.64  232309.80  318456.56  460311.40
normal         1800.0    7381.78    4350.09     51.38   3639.60    7114.75   11134.77   14998.62
rapid_cashout    40.0   23196.02   19793.82   8808.44  12579.68   14499.82   18046.57   76164.77
smurfing         51.0   16611.71   34189.64   7037.27   7742.88    8229.44    8753.36  178709.65
structuring      91.0    9383.76     385.43   8803.46   9025.67    9277.06    9765.04    9974.63


In [7]:
# Log-scale histogram of all amounts
fig = px.histogram(
    tx,
    x="amount",
    nbins=60,
    log_y=True,
    title="Transaction Amount Distribution (log scale)",
    color_discrete_sequence=["#607D8B"],
)
fig.update_layout(xaxis_title="Amount (USD)", yaxis_title="Count (log)")
fig.show()


In [8]:
# Amount distribution by pattern
plot_df = tx.copy()
plot_df["label"] = plot_df["pattern_label"].fillna("normal")
fig = px.box(
    plot_df,
    x="label", y="amount",
    title="Amount Distribution by Pattern",
    color="label",
    log_y=True,
    color_discrete_map={
        "normal": "#607D8B",
        "structuring": "#FF5722",
        "smurfing": "#9C27B0",
        "layering": "#2196F3",
        "rapid_cashout": "#FF9800",
    },
)
fig.update_layout(showlegend=False, xaxis_title="Pattern", yaxis_title="Amount (log)")
fig.show()


## 4. Threshold-Proximity Histogram

This chart shows transactions bucketed near the ,000 BSA CTR threshold.  
The structuring pattern produces a visible spike just below ,000 — the defining signal of structuring.  
**This is one of the most important charts for demonstrating to judges that the detection logic is grounded in real AML typologies.**

In [9]:
# Focus on transactions between ,000 and ,000 (threshold-proximity zone)
THRESHOLD = 10_000
zone = tx[(tx["amount"] >= 8_000) & (tx["amount"] <= 11_000)].copy()
zone["label"] = zone["pattern_label"].fillna("normal")

print(f"Transactions in k-k zone: {len(zone):,}")
print(zone["label"].value_counts().to_string())

fig = px.histogram(
    zone,
    x="amount",
    color="label",
    nbins=60,
    title="Threshold-Proximity Histogram (k–k zone) — Structuring Spike Visible",
    barmode="overlay",
    opacity=0.7,
    color_discrete_map={
        "normal": "#90A4AE",
        "structuring": "#F44336",
        "smurfing": "#9C27B0",
        "layering": "#2196F3",
        "rapid_cashout": "#FF9800",
    },
)
fig.add_vline(
    x=THRESHOLD,
    line_dash="dash",
    line_color="black",
    annotation_text=",000 CTR threshold",
    annotation_position="top right",
)
fig.update_layout(xaxis_title="Amount (USD)", yaxis_title="Count")
fig.show()


Transactions in k-k zone: 464
label
normal           339
structuring       91
smurfing          30
rapid_cashout      4


## 5. Transaction Volume Time Series

In [10]:
tx["date"] = tx["timestamp"].dt.date
daily = tx.groupby(["date", tx["pattern_label"].fillna("normal").rename("label")]).size().reset_index(name="count")

fig = px.line(
    daily,
    x="date", y="count",
    color="label",
    title="Daily Transaction Volume by Label",
    color_discrete_map={
        "normal": "#90A4AE",
        "structuring": "#F44336",
        "smurfing": "#9C27B0",
        "layering": "#2196F3",
        "rapid_cashout": "#FF9800",
    },
)
fig.update_layout(xaxis_title="Date", yaxis_title="Transactions")
fig.show()


## 6. txn_type and Channel Breakdown

In [11]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["txn_type", "channel"])

txn_vc = tx["txn_type"].value_counts()
ch_vc  = tx["channel"].value_counts()

fig.add_trace(go.Bar(x=txn_vc.index, y=txn_vc.values, name="txn_type", marker_color="#42A5F5"), row=1, col=1)
fig.add_trace(go.Bar(x=ch_vc.index,  y=ch_vc.values,  name="channel",  marker_color="#66BB6A"), row=1, col=2)

fig.update_layout(title="txn_type and Channel Distributions", showlegend=False)
fig.show()


## 7. Customer Attribute Distributions

In [12]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["risk_rating", "kyc_status", "customer_type"]
)

for col_idx, col in enumerate(["risk_rating", "kyc_status", "customer_type"], 1):
    vc = cust[col].value_counts()
    fig.add_trace(go.Bar(x=vc.index, y=vc.values, name=col), row=1, col=col_idx)

fig.update_layout(title="Customer Attribute Distributions", showlegend=False)
fig.show()

pep_count = cust["is_pep"].sum()
print(f"PEP customers: {pep_count} ({100*pep_count/len(cust):.1f}%)")
print(f"expected_monthly_volume: mean={cust["expected_monthly_volume"].mean():.0f}, "
      f"median={cust["expected_monthly_volume"].median():.0f}")


PEP customers: 5 (1.9%)
expected_monthly_volume: mean=19821, median=17417


## 8. Missing Value Summary

In [13]:
print("=== Transactions — missing values ===")
tx_null = tx.isnull().sum()
print(tx_null[tx_null > 0].to_string() if tx_null.any() else "  No missing values")
print()
print("Note: pattern_label is null for normal rows (expected).")
print(f"      label_is_laundering null count: {tx["label_is_laundering"].isna().sum()}")
print()
print("=== Customers — missing values ===")
cust_null = cust.isnull().sum()
print(cust_null[cust_null > 0].to_string() if cust_null.any() else "  No missing values")


=== Transactions — missing values ===
label_is_laundering    1800
pattern_label          1800

Note: pattern_label is null for normal rows (expected).
      label_is_laundering null count: 1800

=== Customers — missing values ===
  No missing values


## Summary

| Metric | Value |
|---|---|
| Total transactions | 2,002 |
| Total customers | 270 |
| Normal (unlabelled) | 1,800 (89.9%) |
| Labelled laundering | 202 (10.1%) |
| Patterns: structuring | 91 txns across 10 customers |
| Patterns: smurfing | 51 txns (3 hubs, 24 smurfs) |
| Patterns: rapid_cashout | 40 txns (8 customers) |
| Patterns: layering | 20 txns (5 chains × 4 hops) |
| Schema violations | 0 |

All four required AML patterns are present. Class balance (10% labelled) is intentionally higher than IBM (0.1%) to make pattern detection measurable on the small sample.